# 🏠 EstateIQ — Real Estate Investment Analyzer (Google Colab Edition)

A full-stack ML web app that trains a Gradient Boosting model on the King County House Sales dataset, explains every prediction with SHAP, finds comparable properties via KNN, computes investment metrics (cap rate, gross yield, 5-year projection), and serves an interactive dashboard via Flask + ngrok.

**⚠️ Setup before running Cell 16:**
1. Get a free ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken
2. In Colab, click the 🔑 icon in the left sidebar → **"Add new secret"**
3. Name: `NGROK_TOKEN`, Value: your token, toggle **"Notebook access"** ON

Run the cells top-to-bottom. The final cell prints evaluation metrics.

### Cell 1 — Install Dependencies
Installs Flask (web server), pyngrok (public tunnel), SHAP (model explanations), and the ML stack (scikit-learn, pandas, numpy). The `-q` flag keeps the install output quiet.

In [1]:
!pip install flask pyngrok shap scikit-learn pandas numpy -q
print("✅ Dependencies installed")

✅ Dependencies installed


### Cell 2 — Load Dataset
Downloads the King County House Sales dataset (21,613 rows × 21 columns) directly from GitHub and prints the column list so we can confirm the schema.

In [2]:
import pandas as pd, numpy as np, re, pickle, os, threading, time, json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, r2_score
import shap
from flask import Flask, request, jsonify
from pyngrok import ngrok

# King County House Sales dataset (21,613 rows × 21 columns)
URL = "https://raw.githubusercontent.com/tanlitung/Datasets/master/kc_house_data.csv"
df = pd.read_csv(URL)
print(f"✅ Loaded {len(df):,} rows × {df.shape[1]} columns")
print(df.columns.tolist())

✅ Loaded 21,613 rows × 21 columns
['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']


### Cell 3 — Feature Engineering + Model Training
Creates derived features (`house_age`, `is_renovated`, ratio features, etc.), splits data 80/20, trains a `GradientBoostingRegressor`, and computes the residual standard deviation (used later for confidence intervals).

In [3]:
print("🔧 Engineering features...")

def engineer(d):
    d = d.copy()
    d['house_age']        = 2025 - d['yr_built']
    d['is_renovated']     = (d['yr_renovated'] > 0).astype(int)
    d['yrs_since_renov']  = np.where(d['yr_renovated'] > 0,
                                     2025 - d['yr_renovated'],
                                     d['house_age'])
    d['total_rooms']      = d['bedrooms'] + d['bathrooms']
    d['sqft_ratio']       = d['sqft_living'] / d['sqft_lot'].replace(0, 1)
    d['basement_ratio']   = d['sqft_basement'] / d['sqft_living'].replace(0, 1)
    d['living15_ratio']   = d['sqft_living'] / d['sqft_living15'].replace(0, 1)
    d['lot15_ratio']      = d['sqft_lot'] / d['sqft_lot15'].replace(0, 1)
    d['price_per_sqft']   = d['price'] / d['sqft_living'].replace(0, 1)
    return d

df = engineer(df)

FEATURES = [
    'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement',
    'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long',
    'sqft_living15', 'sqft_lot15',
    'house_age', 'is_renovated', 'yrs_since_renov',
    'total_rooms', 'sqft_ratio', 'basement_ratio',
    'living15_ratio', 'lot15_ratio',
]
TARGET = 'price'

X = df[FEATURES].values
y = df[TARGET].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.9, random_state=42
)
model.fit(Xtr, ytr)

pred_tr = model.predict(Xtr)
pred_te = model.predict(Xte)
residual_std = float(np.std(ytr - pred_tr))

print(f"✅ Model trained")
print(f"   Train R² : {r2_score(ytr, pred_tr):.4f}")
print(f"   Test  R² : {r2_score(yte, pred_te):.4f}")
print(f"   Test  MAE: ${mean_absolute_error(yte, pred_te):,.0f}")
print(f"   Residual σ: ${residual_std:,.0f}")

🔧 Engineering features...
✅ Model trained
   Train R² : 0.9611
   Test  R² : 0.8741
   Test  MAE: $69,489
   Residual σ: $71,330


### Cell 4 — Area Statistics + Comparable Finder
Aggregates per-zipcode price stats (median price, median $/sqft, listing count) and builds a KNN index over key property features so we can find 5 similar properties for any input.

In [4]:
print("📊 Computing area statistics...")

# Area stats per zipcode
area_stats = df.groupby('zipcode').agg(
    area_median_price=('price', 'median'),
    area_mean_price=('price', 'mean'),
    area_median_ppsf=('price_per_sqft', 'median'),
    area_count=('price', 'size'),
).reset_index()

# Scaler for comparable search
comp_features = ['bedrooms', 'bathrooms', 'sqft_living', 'grade', 'condition',
                 'lat', 'long', 'house_age']
comp_scaler = StandardScaler().fit(df[comp_features].values)
comp_matrix = comp_scaler.transform(df[comp_features].values)
comp_index = NearestNeighbors(n_neighbors=6, metric='euclidean').fit(comp_matrix)

def find_comparables(feats):
    row = np.array([[feats[c] for c in comp_features]])
    row_scaled = comp_scaler.transform(row)
    dist, idx = comp_index.kneighbors(row_scaled)
    # drop the first (itself if present in training set)
    comps = df.iloc[idx[0][1:]].copy()
    return comps[['bedrooms','bathrooms','sqft_living','grade',
                  'condition','yr_built','zipcode','price']]

print(f"✅ Area stats for {len(area_stats)} zipcodes")
print(f"✅ Comparable engine ready")

📊 Computing area statistics...
✅ Area stats for 70 zipcodes
✅ Comparable engine ready


### Cell 5 — SHAP Explainer
Builds a `TreeExplainer` for the trained Gradient Boosting model. Takes ~15–30 s. Used to show which features pushed the predicted price up or down.

In [5]:
print("🧠 Building SHAP explainer (this takes ~20s)...")
shap_explainer = shap.TreeExplainer(model)
print("✅ SHAP explainer ready")

🧠 Building SHAP explainer (this takes ~20s)...
✅ SHAP explainer ready


### Cell 6 — Flask App + Analyze Logic
Defines the Flask app, the `build_feature_row()` helper (turns JSON payload → engineered feature row), and the `analyze()` function that produces the full response: prediction, confidence interval, area analysis, comparables, investment metrics, verdict, risk flags, and SHAP contributions.

In [6]:
app = Flask(__name__)

def build_feature_row(p):
    """Turn user payload into a single engineered feature row."""
    yr_built = int(p.get('yr_built', 1990))
    yr_renov = int(p.get('yr_renovated', 0))
    house_age = 2025 - yr_built
    is_renov = 1 if yr_renov > 0 else 0
    yrs_since_renov = (2025 - yr_renov) if is_renov else house_age
    sqft_living = float(p.get('sqft_living', 2000))
    sqft_lot = float(p.get('sqft_lot', 5000))
    sqft_basement = float(p.get('sqft_basement', 0))
    sqft_living15 = float(p.get('sqft_living15', sqft_living))
    sqft_lot15 = float(p.get('sqft_lot15', sqft_lot))
    bedrooms = int(p.get('bedrooms', 3))
    bathrooms = float(p.get('bathrooms', 2))

    row = {
        'bedrooms': bedrooms,
        'bathrooms': bathrooms,
        'sqft_living': sqft_living,
        'sqft_lot': sqft_lot,
        'floors': float(p.get('floors', 1)),
        'waterfront': int(p.get('waterfront', 0)),
        'view': int(p.get('view', 0)),
        'condition': int(p.get('condition', 3)),
        'grade': int(p.get('grade', 7)),
        'sqft_above': float(p.get('sqft_above', sqft_living - sqft_basement)),
        'sqft_basement': sqft_basement,
        'yr_built': yr_built,
        'yr_renovated': yr_renov,
        'zipcode': int(p.get('zipcode', 98001)),
        'lat': float(p.get('lat', 47.5)),
        'long': float(p.get('long', -122.2)),
        'sqft_living15': sqft_living15,
        'sqft_lot15': sqft_lot15,
        'house_age': house_age,
        'is_renovated': is_renov,
        'yrs_since_renov': yrs_since_renov,
        'total_rooms': bedrooms + bathrooms,
        'sqft_ratio': sqft_living / max(sqft_lot, 1),
        'basement_ratio': sqft_basement / max(sqft_living, 1),
        'living15_ratio': sqft_living / max(sqft_living15, 1),
        'lot15_ratio': sqft_lot / max(sqft_lot15, 1),
    }
    return row


def analyze(payload):
    feats = build_feature_row(payload)
    x = np.array([[feats[c] for c in FEATURES]])

    # ---- Prediction + confidence interval ----
    price = float(model.predict(x)[0])
    ci_low  = price - 1.96 * residual_std
    ci_high = price + 1.96 * residual_std
    ppsf = price / max(feats['sqft_living'], 1)

    # ---- Area analysis ----
    zc = feats['zipcode']
    az = area_stats[area_stats['zipcode'] == zc]
    if len(az):
        area_med_price = float(az['area_median_price'].iloc[0])
        area_med_ppsf  = float(az['area_median_ppsf'].iloc[0])
        area_count     = int(az['area_count'].iloc[0])
        zip_prices = df[df['zipcode'] == zc]['price'].values
        percentile = float((zip_prices < price).mean() * 100)
    else:
        area_med_price = float(df['price'].median())
        area_med_ppsf  = float(df['price_per_sqft'].median())
        area_count = 0
        percentile = 50.0

    ppsf_vs_area = (ppsf / area_med_ppsf - 1) * 100 if area_med_ppsf else 0

    # ---- Comparable properties ----
    comps = find_comparables(feats)
    comp_list = comps.to_dict(orient='records')
    comp_median = float(np.median([c['price'] for c in comp_list]))

    # ---- Investment metrics ----
    monthly_rent   = price * 0.004              # ~4.8% annual yield heuristic
    annual_rent    = monthly_rent * 12
    gross_yield    = (annual_rent / price) * 100
    annual_tax     = price * 0.0092             # King County ~0.92%
    net_income     = annual_rent - annual_tax
    cap_rate       = (net_income / price) * 100
    five_yr_value  = price * (1.04 ** 5)

    # ---- Deal verdict ----
    if price < comp_median * 0.92:
        verdict = "UNDERVALUED"
        verdict_color = "green"
    elif price > comp_median * 1.08:
        verdict = "OVERVALUED"
        verdict_color = "red"
    else:
        verdict = "FAIR PRICE"
        verdict_color = "amber"

    # ---- Risk flags ----
    flags = []
    if ppsf_vs_area > 25:
        flags.append(f"Price per sqft is {ppsf_vs_area:.0f}% above zipcode median")
    if feats['house_age'] > 50 and not feats['is_renovated']:
        flags.append(f"Property is {feats['house_age']} years old with no renovation")
    if feats['condition'] <= 2:
        flags.append("Low condition score (≤2)")
    if feats['grade'] <= 5:
        flags.append("Below-average construction grade")
    if feats['sqft_living'] < 1000:
        flags.append("Small living area (< 1000 sqft)")
    if feats['bedrooms'] >= 6:
        flags.append("Unusually high bedroom count — verify layout")
    if not flags:
        flags.append("No major risk flags detected")

    # ---- SHAP contributions (top 8) ----
    sv = shap_explainer.shap_values(x)[0]
    contributions = sorted(
        zip(FEATURES, sv),
        key=lambda t: -abs(t[1])
    )[:8]
    shap_list = [
        {'feature': f, 'value': feats[f], 'impact': float(round(v))}
        for f, v in contributions
    ]

    return {
        'price': round(price),
        'range_low': round(ci_low),
        'range_high': round(ci_high),
        'price_per_sqft': round(ppsf),
        'area': {
            'zipcode': zc,
            'median_price': round(area_med_price),
            'median_ppsf': round(area_med_ppsf),
            'count': area_count,
            'percentile': round(percentile, 1),
            'ppsf_vs_area': round(ppsf_vs_area, 1),
        },
        'comparables': comp_list,
        'comparable_median': round(comp_median),
        'investment': {
            'monthly_rent': round(monthly_rent),
            'annual_rent': round(annual_rent),
            'gross_yield': round(gross_yield, 2),
            'annual_tax': round(annual_tax),
            'cap_rate': round(cap_rate, 2),
            'five_yr_value': round(five_yr_value),
        },
        'verdict': verdict,
        'verdict_color': verdict_color,
        'risk_flags': flags,
        'shap': shap_list,
    }


@app.route('/')
def home():
    return HTML_PAGE

@app.route('/api/analyze', methods=['POST'])
def api_analyze():
    payload = request.get_json() or {}
    try:
        result = analyze(payload)
        return jsonify(result)
    except Exception as e:
        return jsonify({'error': str(e)}), 400

print("✅ Flask app + /api/analyze endpoint defined")

✅ Flask app + /api/analyze endpoint defined


### Cell 7 — Frontend HTML
Defines the single-page dashboard as a raw Python string (`HTML_PAGE`). Includes glassmorphism styling, animated background blobs, a property input form, results panel, comparables table, and a Chart.js bar chart for SHAP values.

In [11]:
HTML_PAGE = r'''<!DOCTYPE html>
<html lang='en'>
<head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1.0'>
<title>EstateIQ — Real Estate Investment Analyzer</title>
<link href='https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&family=JetBrains+Mono:wght@500&display=swap' rel='stylesheet'>
<script src='https://cdn.jsdelivr.net/npm/chart.js'></script>
<style>
* { margin: 0; padding: 0; box-sizing: border-box; }
:root {
  --card: rgba(255,255,255,0.04);
  --border: rgba(255,255,255,0.08);
  --text: #f1f5f9;
  --dim: #94a3b8;
  --indigo: #6366f1;
  --pink: #ec4899;
  --cyan: #06b6d4;
  --green: #10b981;
  --red: #ef4444;
  --amber: #f59e0b;
}
body {
  font-family: 'Inter', sans-serif;
  background: radial-gradient(ellipse at top, #0f172a 0%, #020617 60%);
  min-height: 100vh;
  color: var(--text);
  padding: 22px 28px;
  overflow-x: hidden;
}
body::before, body::after {
  content: '';
  position: fixed;
  width: 700px; height: 700px;
  border-radius: 50%;
  pointer-events: none;
  z-index: 0;
  animation: float 26s ease-in-out infinite;
}
body::before {
  top: -250px; left: -250px;
  background: radial-gradient(circle, rgba(99,102,241,0.14), transparent 70%);
}
body::after {
  bottom: -250px; right: -250px;
  background: radial-gradient(circle, rgba(6,182,212,0.10), transparent 70%);
  animation-direction: reverse;
}
@keyframes float {
  0%, 100% { transform: translate(0, 0); }
  50% { transform: translate(70px, 50px); }
}
.container { max-width: 1560px; margin: 0 auto; position: relative; z-index: 1; }

/* ---------- HEADER : one horizontal strip ---------- */
header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 24px;
  margin-bottom: 14px;
  text-align: left;
}
header h1 {
  font-size: 32px; font-weight: 900;
  background: linear-gradient(135deg, #a5b4fc 0%, #67e8f9 50%, #f9a8d4 100%);
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  background-clip: text; letter-spacing: -1px; line-height: 1.1;
}
header p { color: var(--dim); font-size: 12.5px; margin-top: 3px; }
.badge {
  display: inline-block; padding: 5px 14px;
  background: rgba(16,185,129,0.12); border: 1px solid rgba(16,185,129,0.3);
  color: #6ee7b7; border-radius: 20px; font-size: 11px; font-weight: 700;
  letter-spacing: 1px; flex-shrink: 0; white-space: nowrap;
}
.badge::before { content: '\25CF'; margin-right: 6px; animation: pulse 2s infinite; }
@keyframes pulse { 0%,100% { opacity: 1; } 50% { opacity: 0.3; } }

/* ---------- LAYOUT ---------- */
.layout { display: block; }

.panel {
  background: var(--card);
  border: 1px solid var(--border);
  border-radius: 14px;
  padding: 16px 18px;
  backdrop-filter: blur(20px);
  box-shadow: 0 8px 40px rgba(0,0,0,0.3);
}
.panel h2 { font-size: 15px; margin-bottom: 2px; font-weight: 700; }
.panel .subtitle { color: var(--dim); font-size: 11px; }

/* ---------- FORM : wide horizontal band ---------- */
.form-panel {
  display: grid;
  grid-template-columns: 215px 1.15fr 1.25fr 0.95fr;
  gap: 20px 22px;
  align-items: start;
}
.form-head { display: flex; flex-direction: column; align-self: stretch; }
.form-head .subtitle { margin-top: 2px; }
.form-head .btn { margin-top: auto; }

.section-label {
  font-size: 9.5px; font-weight: 800; color: var(--indigo);
  text-transform: uppercase; letter-spacing: 1.1px;
  margin: 14px 0 8px; padding-bottom: 5px;
  border-bottom: 1px solid rgba(99,102,241,0.15);
}
.form-col > .section-label:first-child { margin-top: 0; }

.field { margin-bottom: 8px; }
.field label {
  display: block; font-size: 10px; font-weight: 600;
  color: var(--dim); margin-bottom: 3px;
}
.field input, .field select {
  width: 100%; padding: 7px 9px;
  border-radius: 7px; border: 1px solid var(--border);
  background: rgba(0,0,0,0.35); color: var(--text);
  font-family: inherit; font-size: 12.5px; outline: none;
  transition: all 0.2s;
}
.field input:focus, .field select:focus {
  border-color: var(--indigo);
  box-shadow: 0 0 0 3px rgba(99,102,241,0.1);
}
.field select option { background: #0f172a; }
.grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }
.grid-3 { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 8px; }
.btn {
  background: linear-gradient(135deg, var(--indigo), var(--cyan));
  border: none; color: white; width: 100%;
  padding: 12px; border-radius: 9px;
  font-family: inherit; font-size: 13px; font-weight: 700;
  cursor: pointer; transition: all 0.2s;
  margin-top: 16px; letter-spacing: 0.3px;
  box-shadow: 0 4px 20px rgba(99,102,241,0.3);
}
.btn:hover:not(:disabled) { transform: translateY(-2px); box-shadow: 0 6px 28px rgba(99,102,241,0.5); }
.btn:disabled { opacity: 0.6; cursor: not-allowed; }

/* ---------- RESULTS : landscape dashboard grid ---------- */
#results { display: none; }
#results.show {
  display: grid;
  grid-template-columns: 1fr 1.15fr 1fr;
  gap: 14px;
  margin-top: 14px;
  animation: fadeUp 0.4s ease;
}
@keyframes fadeUp { from { opacity: 0; transform: translateY(10px); } to { opacity: 1; transform: translateY(0); } }

/* hero band spans the full width */
.panel-hero {
  grid-column: 1 / -1;
  display: flex;
  align-items: center;
  gap: 18px;
  padding: 14px 18px;
}
.price-hero {
  flex: 0 0 300px;
  background: linear-gradient(135deg, rgba(99,102,241,0.15), rgba(6,182,212,0.10));
  border: 1px solid rgba(99,102,241,0.25);
  border-radius: 12px; padding: 14px 18px; text-align: center;
}
.price-hero .label {
  font-size: 10px; font-weight: 700; color: var(--dim);
  text-transform: uppercase; letter-spacing: 1.4px;
}
.price-hero .price {
  font-size: 34px; font-weight: 900; letter-spacing: -1.2px;
  background: linear-gradient(135deg, #a5b4fc, #67e8f9);
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  margin: 3px 0;
}
.price-hero .range { color: var(--dim); font-size: 11.5px; font-family: 'JetBrains Mono', monospace; }
.verdict-badge {
  display: inline-block; padding: 5px 14px; border-radius: 20px;
  font-size: 11px; font-weight: 800; letter-spacing: 1px;
  margin-top: 9px;
}
.verdict-badge.green { background: rgba(16,185,129,0.15); color: #6ee7b7; border: 1px solid rgba(16,185,129,0.3); }
.verdict-badge.red { background: rgba(239,68,68,0.15); color: #fca5a5; border: 1px solid rgba(239,68,68,0.3); }
.verdict-badge.amber { background: rgba(245,158,11,0.15); color: #fcd34d; border: 1px solid rgba(245,158,11,0.3); }

.hero-side { flex: 1; min-width: 0; }
.area-display { font-size: 12.5px; color: var(--dim); line-height: 1.6; }
.area-display strong { color: var(--text); }

.metrics-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 10px; }
#investment-grid { grid-template-columns: repeat(2, 1fr); }

.metric-card {
  background: rgba(255,255,255,0.03); border: 1px solid var(--border);
  border-radius: 10px; padding: 10px 12px;
}
.metric-card .label {
  font-size: 9.5px; font-weight: 700; color: var(--dim);
  text-transform: uppercase; letter-spacing: 0.7px;
}
.metric-card .value {
  font-size: 17px; font-weight: 700; margin-top: 3px;
  font-family: 'JetBrains Mono', monospace;
}
.section-title {
  font-size: 12px; font-weight: 700; color: var(--text);
  margin: 14px 0 8px; display: flex; align-items: center; gap: 8px;
}
.panel > .section-title:first-child { margin-top: 0; }
.section-title::before {
  content: ''; width: 3px; height: 13px;
  background: linear-gradient(180deg, var(--indigo), var(--cyan));
  border-radius: 2px;
}

table { width: 100%; border-collapse: collapse; font-size: 11.5px; }
th { text-align: left; padding: 6px 6px; color: var(--dim); font-weight: 600; font-size: 9.5px; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 1px solid var(--border); }
td { padding: 6px 6px; border-bottom: 1px solid rgba(255,255,255,0.04); font-family: 'JetBrains Mono', monospace; font-size: 11.5px; }
tr:hover td { background: rgba(99,102,241,0.04); }

.flags { list-style: none; }
.flags li {
  padding: 6px 10px; margin-bottom: 5px;
  background: rgba(245,158,11,0.08);
  border-left: 3px solid var(--amber);
  border-radius: 6px; font-size: 11.5px; color: #fcd34d;
}
.flags li.ok {
  background: rgba(16,185,129,0.08);
  border-left-color: var(--green); color: #6ee7b7;
}

canvas { max-height: 240px; }
footer { text-align: center; margin-top: 18px; color: var(--dim); font-size: 10.5px; }
.spinner {
  display: inline-block; width: 14px; height: 14px;
  border: 2px solid rgba(255,255,255,0.3);
  border-top-color: white; border-radius: 50%;
  animation: spin 0.6s linear infinite;
  vertical-align: middle; margin-right: 8px;
}
@keyframes spin { to { transform: rotate(360deg); } }

/* ---------- RESPONSIVE FALLBACKS ---------- */
@media (max-width: 1250px) {
  .form-panel { grid-template-columns: 1fr 1fr; }
  .form-head { grid-column: 1 / -1; }
  .form-head .btn { margin-top: 12px; }
  #results.show { grid-template-columns: 1fr 1fr; }
  .panel-hero { flex-direction: column; align-items: stretch; }
  .price-hero { flex: none; }
}
@media (max-width: 780px) {
  body { padding: 16px 14px; }
  .form-panel { grid-template-columns: 1fr; }
  #results.show { grid-template-columns: 1fr; }
  header { flex-direction: column; align-items: flex-start; gap: 10px; }
}
</style>
</head>
<body>
<div class='container'>
  <header>
    <div>
      <h1>EstateIQ</h1>
      <p>Real Estate Investment Analyzer — powered by Gradient Boosting + SHAP</p>
    </div>
    <span class='badge'>LIVE MODEL</span>
  </header>

  <div class='layout'>
    <!-- ============ INPUT FORM (horizontal band) ============ -->
    <div class='panel form-panel'>

      <div class='form-head'>
        <h2>Property Details</h2>
        <p class='subtitle'>Fill in the property attributes below</p>
        <button class='btn' id='analyze-btn' onclick='analyze()'>Analyze Investment</button>
      </div>

      <div class='form-col'>
        <div class='section-label'>Size & Layout</div>
        <div class='grid-3'>
          <div class='field'><label>Bedrooms</label><input type='number' id='bedrooms' value='3'></div>
          <div class='field'><label>Bathrooms</label><input type='number' step='0.25' id='bathrooms' value='2'></div>
          <div class='field'><label>Floors</label><input type='number' step='0.5' id='floors' value='1'></div>
        </div>
        <div class='grid-2'>
          <div class='field'><label>Sqft Living</label><input type='number' id='sqft_living' value='2000'></div>
          <div class='field'><label>Sqft Lot</label><input type='number' id='sqft_lot' value='5000'></div>
        </div>
        <div class='grid-2'>
          <div class='field'><label>Sqft Above</label><input type='number' id='sqft_above' value='2000'></div>
          <div class='field'><label>Sqft Basement</label><input type='number' id='sqft_basement' value='0'></div>
        </div>
        <div class='grid-2'>
          <div class='field'><label>Sqft Living (15 neighbors)</label><input type='number' id='sqft_living15' value='2000'></div>
          <div class='field'><label>Sqft Lot (15 neighbors)</label><input type='number' id='sqft_lot15' value='5000'></div>
        </div>
      </div>

      <div class='form-col'>
        <div class='section-label'>Quality & Condition</div>
        <div class='grid-3'>
          <div class='field'><label>Grade (1-13)</label><input type='number' id='grade' value='7'></div>
          <div class='field'><label>Condition (1-5)</label><input type='number' id='condition' value='3'></div>
          <div class='field'><label>View (0-4)</label><input type='number' id='view' value='0'></div>
        </div>
        <div class='field'><label>Waterfront</label>
          <select id='waterfront'><option value='0'>No</option><option value='1'>Yes</option></select>
        </div>

        <div class='section-label'>Age & Renovation</div>
        <div class='grid-2'>
          <div class='field'><label>Year Built</label><input type='number' id='yr_built' value='1990'></div>
          <div class='field'><label>Year Renovated (0 if never)</label><input type='number' id='yr_renovated' value='0'></div>
        </div>
      </div>

      <div class='form-col'>
        <div class='section-label'>Location</div>
        <div class='field'><label>Zipcode</label><input type='number' id='zipcode' value='98001'></div>
        <div class='grid-2'>
          <div class='field'><label>Latitude</label><input type='number' step='0.0001' id='lat' value='47.5112'></div>
          <div class='field'><label>Longitude</label><input type='number' step='0.0001' id='long' value='-122.2570'></div>
        </div>
      </div>

    </div>

    <!-- ============ RESULTS ============ -->
    <div id='results'>

      <!-- Full-width hero band -->
      <div class='panel panel-hero'>
        <div class='price-hero'>
          <div class='label'>Estimated Market Value</div>
          <div class='price' id='price-display'>—</div>
          <div class='range' id='range-display'>—</div>
          <div id='verdict-display'></div>
        </div>
        <div class='hero-side'>
          <div class='metrics-grid' id='metrics-grid'></div>
          <div class='area-display' id='area-display'></div>
        </div>
      </div>

      <!-- Column 1 : investment + risk -->
      <div class='panel'>
        <div class='section-title'>Investment Metrics</div>
        <div class='metrics-grid' id='investment-grid'></div>

        <div class='section-title'>Risk Flags</div>
        <ul class='flags' id='flags-list'></ul>
      </div>

      <!-- Column 2 : comparables -->
      <div class='panel'>
        <div class='section-title'>Comparable Properties</div>
        <div style='overflow-x:auto;'>
          <table id='comparables-table'>
            <thead><tr><th>Bed</th><th>Bath</th><th>Sqft</th><th>Grade</th><th>Yr</th><th>Zip</th><th>Price</th></tr></thead>
            <tbody></tbody>
          </table>
        </div>
      </div>

      <!-- Column 3 : SHAP -->
      <div class='panel'>
        <div class='section-title'>What Drives This Price (SHAP)</div>
        <canvas id='shap-chart'></canvas>
      </div>

    </div>
  </div>

  <footer>EstateIQ — Gradient Boosting on King County House Sales · SHAP explanations · served via ngrok</footer>
</div>

<script>
var shapChart = null;

function fmt(n) { return '$' + Math.round(n).toLocaleString(); }

function analyze() {
  var btn = document.getElementById('analyze-btn');
  btn.disabled = true;
  btn.innerHTML = `<span class='spinner'></span>Analyzing...`;

  var payload = {
    bedrooms: +document.getElementById('bedrooms').value,
    bathrooms: +document.getElementById('bathrooms').value,
    sqft_living: +document.getElementById('sqft_living').value,
    sqft_lot: +document.getElementById('sqft_lot').value,
    floors: +document.getElementById('floors').value,
    waterfront: +document.getElementById('waterfront').value,
    view: +document.getElementById('view').value,
    condition: +document.getElementById('condition').value,
    grade: +document.getElementById('grade').value,
    sqft_above: +document.getElementById('sqft_above').value,
    sqft_basement: +document.getElementById('sqft_basement').value,
    yr_built: +document.getElementById('yr_built').value,
    yr_renovated: +document.getElementById('yr_renovated').value,
    zipcode: +document.getElementById('zipcode').value,
    lat: +document.getElementById('lat').value,
    long: +document.getElementById('long').value,
    sqft_living15: +document.getElementById('sqft_living15').value,
    sqft_lot15: +document.getElementById('sqft_lot15').value,
  };

  fetch('/api/analyze', {
    method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify(payload)
  }).then(r => r.json()).then(function(d) {
    if (d.error) { alert('Error: ' + d.error); btn.disabled = false; btn.innerHTML = 'Analyze Investment'; return; }
    renderResults(d);
    btn.disabled = false;
    btn.innerHTML = 'Analyze Investment';
  }).catch(function(e) {
    alert('Error: ' + e.message);
    btn.disabled = false; btn.innerHTML = 'Analyze Investment';
  });
}

function renderResults(d) {
  document.getElementById('results').classList.add('show');

  document.getElementById('price-display').textContent = fmt(d.price);
  document.getElementById('range-display').textContent =
    fmt(d.range_low) + '  —  ' + fmt(d.range_high) + '  (95% CI)';

  var v = document.getElementById('verdict-display');
  v.innerHTML = `<span class='verdict-badge ${d.verdict_color}'>${d.verdict}</span>`;

  document.getElementById('metrics-grid').innerHTML =
    card('Price / Sqft', '$' + d.price_per_sqft) +
    card('Comparable Median', fmt(d.comparable_median)) +
    card('Zipcode Percentile', d.area.percentile + '%');

  document.getElementById('area-display').innerHTML =
    `Zipcode <strong>${d.area.zipcode}</strong> · Median price <strong>${fmt(d.area.median_price)}</strong> ` +
    `· Median $/sqft <strong>$${d.area.median_ppsf}</strong> · Listings <strong>${d.area.count}</strong><br>` +
    `This property is priced <strong>${d.area.ppsf_vs_area > 0 ? '+' : ''}${d.area.ppsf_vs_area}%</strong> ` +
    `per sqft vs the zipcode median.`;

  var tbody = document.querySelector('#comparables-table tbody');
  tbody.innerHTML = '';
  d.comparables.forEach(function(c) {
    tbody.innerHTML += `<tr>
      <td>${c.bedrooms}</td><td>${c.bathrooms}</td><td>${c.sqft_living}</td>
      <td>${c.grade}</td><td>${c.yr_built}</td><td>${c.zipcode}</td>
      <td style='color:#6ee7b7;'>${fmt(c.price)}</td>
    </tr>`;
  });

  document.getElementById('investment-grid').innerHTML =
    card('Monthly Rent (est.)', fmt(d.investment.monthly_rent)) +
    card('Gross Yield', d.investment.gross_yield + '%') +
    card('Cap Rate', d.investment.cap_rate + '%') +
    card('Annual Tax (est.)', fmt(d.investment.annual_tax)) +
    card('Annual Rent', fmt(d.investment.annual_rent)) +
    card('5-Year Value', fmt(d.investment.five_yr_value));

  document.getElementById('flags-list').innerHTML = d.risk_flags.map(function(f) {
    var isOk = f.indexOf('No major') === 0;
    return `<li class='${isOk ? 'ok' : ''}'>${f}</li>`;
  }).join('');

  renderShap(d.shap);
}

function card(label, value) {
  return `<div class='metric-card'><div class='label'>${label}</div><div class='value'>${value}</div></div>`;
}

function renderShap(shapData) {
  var labels = shapData.map(function(s) { return s.feature; });
  var values = shapData.map(function(s) { return s.impact; });
  var colors = values.map(function(v) { return v > 0 ? 'rgba(16,185,129,0.7)' : 'rgba(239,68,68,0.7)'; });

  if (shapChart) shapChart.destroy();
  var ctx = document.getElementById('shap-chart').getContext('2d');
  shapChart = new Chart(ctx, {
    type: 'bar',
    data: {
      labels: labels,
      datasets: [{
        data: values,
        backgroundColor: colors,
        borderRadius: 4,
      }]
    },
    options: {
      indexAxis: 'y',
      responsive: true,
      plugins: {
        legend: { display: false },
        tooltip: {
          callbacks: {
            label: function(c) {
              return (c.raw >= 0 ? '+' : '') + '$' + Math.round(c.raw).toLocaleString();
            }
          }
        }
      },
      scales: {
        x: {
          ticks: { color: '#94a3b8', callback: function(v) { return (v >= 0 ? '+' : '') + '$' + Math.round(v/1000) + 'k'; } },
          grid: { color: 'rgba(255,255,255,0.05)' }
        },
        y: { ticks: { color: '#cbd5e1', font: { size: 11 } }, grid: { display: false } }
      }
    }
  });
}
</script>
</body>
</html>'''

print("✅ Frontend HTML defined")

✅ Frontend HTML defined


### Cell 8 — Launch Flask + ngrok (Colab-ready)
**This is the critical cell.** It imports `userdata` from `google.colab`, reads your `NGROK_TOKEN` secret, kills any stale process on port 5000, starts Flask in a background daemon thread, and opens an ngrok tunnel. Prints the public URL.

In [8]:
# ============================================================
# CELL 8 — Launch Flask + ngrok (Colab-ready)
# ============================================================
from google.colab import userdata  # <-- Colab secret access
import os, signal, subprocess, time, threading

# ---- Load ngrok token from Colab Secrets ----
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
except Exception as e:
    NGROK_TOKEN = None
    print(f"⚠️  Could not read Colab secret: {e}")

if not NGROK_TOKEN:
    raise SystemExit(
        "⚠️  NGROK_TOKEN secret not set!\n"
        "   1. Get a free token: https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "   2. In Colab: 🔑 icon (left sidebar) → Add new secret → Name: NGROK_TOKEN → toggle 'Notebook access' ON\n"
        "   3. Re-run this cell."
    )

ngrok.set_auth_token(NGROK_TOKEN)
print("🔑 ngrok auth token loaded from Colab secrets")

# ---- Step 1: kill anything already on port 5000 ----
def kill_port(port):
    """Kill any process listening on the given port (Linux/Colab)."""
    try:
        result = subprocess.run(
            ['bash', '-c', f"lsof -ti:{port} 2>/dev/null || true"],
            capture_output=True, text=True
        )
        pids = result.stdout.strip().split()
        for pid in pids:
            try:
                os.kill(int(pid), signal.SIGKILL)
                print(f"   🔪 Killed stale process PID {pid} on port {port}")
            except Exception as e:
                print(f"   ⚠️  Could not kill PID {pid}: {e}")
    except Exception as e:
        print(f"   ⚠️  kill_port skipped: {e}")

PORT = 5000
kill_port(PORT)
time.sleep(1)

# ---- Step 2: start Flask in background thread ----
def run_flask():
    app.run(host='0.0.0.0', port=PORT, use_reloader=False, debug=False, threaded=True)

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(3)
print(f"✅ Flask running on http://localhost:{PORT}")

# ---- Step 3: create ngrok tunnel and extract the clean URL string ----
try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url
print(f"🔍 public_url type: {type(public_url).__name__}")
print(f"🔍 public_url value: {public_url}")

print()
print("=" * 64)
print(f"🌐  LIVE URL:  {public_url}")
print("=" * 64)
print()
print("👉 Open the URL in your browser")
print("👉 Stop the tunnel later: from pyngrok import ngrok; ngrok.kill()")

🔑 ngrok auth token loaded from Colab secrets
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Flask running on http://localhost:5000
🔍 public_url type: str
🔍 public_url value: https://pauper-consent-handclap.ngrok-free.dev

🌐  LIVE URL:  https://pauper-consent-handclap.ngrok-free.dev

👉 Open the URL in your browser
👉 Stop the tunnel later: from pyngrok import ngrok; ngrok.kill()


### Cell 9 — Test the API
Sends a sample property (4 bed / 2.5 bath / 2500 sqft in Bellevue 98004) to the live ngrok URL and prints the returned prediction, verdict, yield, cap rate, risk flags, and top SHAP drivers.

In [9]:
import requests, json

# public_url is now a plain string — this will work
url = public_url.rstrip('/') + '/api/analyze'
print(f"Testing: {url}\n")

test_property = {
    "bedrooms": 4, "bathrooms": 2.5, "sqft_living": 2500, "sqft_lot": 6000,
    "floors": 2, "waterfront": 0, "view": 2, "condition": 4, "grade": 8,
    "sqft_above": 2500, "sqft_basement": 0, "yr_built": 1995,
    "yr_renovated": 2015, "zipcode": 98004, "lat": 47.6279, "long": -122.2030,
    "sqft_living15": 2400, "sqft_lot15": 5800,
}

r = requests.post(url, json=test_property, timeout=30)
d = r.json()

if 'error' in d:
    print("❌ API error:", d['error'])
else:
    print(f"Price        : ${d['price']:,}")
    print(f"Range        : ${d['range_low']:,} — ${d['range_high']:,}")
    print(f"$/sqft       : ${d['price_per_sqft']}")
    print(f"Verdict      : {d['verdict']}")
    print(f"Gross yield  : {d['investment']['gross_yield']}%")
    print(f"Cap rate     : {d['investment']['cap_rate']}%")
    print(f"Monthly rent : ${d['investment']['monthly_rent']:,}")
    print(f"Risk flags   : {d['risk_flags']}")
    print(f"\nTop SHAP drivers:")
    for s in d['shap'][:5]:
        print(f"   {s['feature']:20s} impact = ${s['impact']:+,}")

Testing: https://pauper-consent-handclap.ngrok-free.dev/api/analyze



INFO:werkzeug:127.0.0.1 - - [02/Oct/2026 16:54:40] "POST /api/analyze HTTP/1.1" 200 -


Price        : $1,224,388
Range        : $1,084,582 — $1,364,194
$/sqft       : $490
Verdict      : OVERVALUED
Gross yield  : 4.8%
Cap rate     : 3.88%
Monthly rent : $4,898
Risk flags   : ['No major risk flags detected']

Top SHAP drivers:
   lat                  impact = $+274,780.0
   zipcode              impact = $+163,083.0
   view                 impact = $+76,579.0
   long                 impact = $+57,682.0
   sqft_above           impact = $+43,592.0


### Cell 10 — Model Evaluation Metrics
Prints R², MAE, MAPE, median absolute error, and the percentage of test predictions within ±10% and ±20% of actual price.

In [10]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(Xte)
mae = mean_absolute_error(yte, preds)
r2  = r2_score(yte, preds)
mape = np.mean(np.abs((yte - preds) / yte)) * 100

print(f"Test set size    : {len(yte):,}")
print(f"R²               : {r2:.4f}")
print(f"MAE              : ${mae:,.0f}")
print(f"MAPE             : {mape:.2f}%")
print(f"Median abs error : ${np.median(np.abs(yte - preds)):,.0f}")
print(f"Within ±10%      : {(np.abs(yte - preds) / yte < 0.10).mean()*100:.1f}%")
print(f"Within ±20%      : {(np.abs(yte - preds) / yte < 0.20).mean()*100:.1f}%")

Test set size    : 4,323
R²               : 0.8741
MAE              : $69,489
MAPE             : 12.76%
Median abs error : $40,129
Within ±10%      : 54.4%
Within ±20%      : 82.0%
